<a href="https://colab.research.google.com/github/antoinewrd1/Antoine-Ward/blob/main/ai_chatbot_final_python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Created on Thu Apr  9 18:52:31 2026

@author: Antoine Ward
"""
import json
from datetime import datetime
from typing import List, Dict

class ConversationManager:

    """
    Manages the state, history, and context of a user conversation

    """

    def __init__(self, user_id: str, max_history: int=10):
        """
        Args:
            user_id: A unique identifier for the user.
            max_history: The maximum number of message exchanges to retain.
            """
        self.user_id = user_id
        self.max_history = max_history
        self.history: List[Dict] = []

        #Stores summarized older messages
        self.summary: str = ""

        #Track metadata such as start time or total tokens used
        self.metadata = {"session_active": True, "turn_count": 0, "total_tokens_estimated": 0, "start_time": datetime.now().isoformat()}

    def add_message(self, role: str, content: str):
        """ Add a message with timestamp and token tracking."""

        message = {
            "role": role,
            "content": content,
            "timestamp": datetime.now().isoformat()
        }

        """ Helper to append a message and maintain the history limit."""

        self.history.append(message)

        self.metadata["turn_count"] += 1

        #Estimate tokens (simple approximation: 1 token is approximately 4 chars)
        tokens = len(content) //4
        self.metadata["total_tokens_estimated"] += tokens

        #Prune history if needed
        if len(self.history) > self.max_history:
            self._summarize_and_prune()

    def get_history(self) -> List[Dict]:
      """Return current conversation history. """
      return self.history

    def get_context(self) -> str:
      """Return full context including summary + recent history."""
      context = ""

      if self.summary:
        context += f"Summary of earlier conversation:\n{self.summary}\n\n"

      for msg in self.history:
        context += f"{msg['role']}: {msg['content']}\n"

      return context.strip()

    def _summarize_and_prune(self):
      """Summarize oldest messages and remove them."""

      #Take oldest message
      oldest = self.history.pop(0)

      #Simple summarization (can replace with LLM later)
      summary_piece = f"{oldest['role']} said '{oldest['content'][:50]}...'"

      if self.summary:
        self.summary += " | " + summary_piece
      else:
        self.summary = summary_piece

    def save_to_file(self, filepath:str):
      """Save session to a JSON file."""

      data = {
          "user_id": self.user_id,
          "history": self.history,
          "summary": self.summary,
          "metadata": self.metadata

      }

      with open(filepath, "w") as f:
        json.dump(data, f, indent=4)

    @classmethod
    def load_from_file(cls, filepath: str):
      """Load session from a JSON file."""
      with open(filepath, "r") as f:
        data = json.load(f)

      obj = cls(user_id=data["user_id"])
      obj.history = data["history"]
      obj.summary = data["summary"]
      obj.metadata = data["metadata"]

      return obj

  #---------------------Test Code----------------------#


if __name__ == "__main__":

  chat_session = ConversationManager(user_id = "user_123", max_history=3)

  messages = [
      ("user", "Hi there!"),
      ("assistant", "Hello! How can I help you today?"),
      ("user", "what is the weather like?"),
      ("assistant", "It's sunny and 50 degrees Fahrenheit"),
      ("user", "Cool, thanks!")
      ]

print(f"--- Starting Test for User: {chat_session.user_id} ---")

for role, text in messages:

    chat_session.add_message(role, text)

    print("\n--- Current Context ---")
    print(chat_session.get_context())

    print("\n--- Metadata ---")
    print(chat_session.metadata)

#Save session
chat_session.save_to_file("chat_session.json")

#Load session
loaded_session = ConversationManager.load_from_file("chat_session.json")

print("\n--- Loaded Session Context ---")
print(loaded_session.get_context())


